# Stepwise Molecular Assembly Simulation

**Goal:** Simulate 6 different molecules diffusing on a 2D surface and assembling in a stepwise manner. Each molecule is labeled with a unique dye, and we visualize the assembly process using ground truth RGB rendering.

**Assembly Process:**
- Molecules A, B, C, D, E, F labeled with 6 different spectral dyes
- Stepwise assembly: A+B → AB, AB+C → ABC, ABC+D → ABCD, etc.
- Visualize as colors combine to form multi-color complexes

**Output:**
- Ground truth RGB video showing assembly dynamics
- Color combinations reveal binding states

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from DiffusionSimulation import (
    DiffusionSimulator2D,
    CameraAdapter
)

# Set random seed for reproducibility
np.random.seed(42)

print("Imports successful!")

Imports successful!


## 1. Define Binding Kinetics Matrix

We'll create a binding matrix for stepwise assembly:
- A + B ⇌ AB (fast)
- AB + C ⇌ ABC (moderate)
- ABC + D ⇌ ABCD (moderate)
- ABCD + E ⇌ ABCDE (slow)
- ABCDE + F ⇌ ABCDEF (slow)

This creates a hierarchical assembly pathway.

In [2]:
# Define 6 molecule types (colors)
# We'll use spectral colors spanning the visible spectrum
molecules = {
    'A': {'color': 'Blue',   'A_R': 0.05, 'A_G': 0.15, 'A_B': 0.80},  # Deep blue
    'B': {'color': 'Cyan',   'A_R': 0.10, 'A_G': 0.50, 'A_B': 0.40},  # Cyan
    'C': {'color': 'Green',  'A_R': 0.15, 'A_G': 0.75, 'A_B': 0.10},  # Green
    'D': {'color': 'Yellow', 'A_R': 0.50, 'A_G': 0.45, 'A_B': 0.05},  # Yellow
    'E': {'color': 'Orange', 'A_R': 0.70, 'A_G': 0.25, 'A_B': 0.05},  # Orange
    'F': {'color': 'Red',    'A_R': 0.80, 'A_G': 0.15, 'A_B': 0.05},  # Red
}

# Create color list for binding matrix
colors = ['Blue', 'Cyan', 'Green', 'Yellow', 'Orange', 'Red']
n_species = len(colors)

print("Molecule spectral profiles:")
for name, props in molecules.items():
    print(f"  {name} ({props['color']:6s}): R={props['A_R']:.2f}, G={props['A_G']:.2f}, B={props['A_B']:.2f}")

Molecule spectral profiles:
  A (Blue  ): R=0.05, G=0.15, B=0.80
  B (Cyan  ): R=0.10, G=0.50, B=0.40
  C (Green ): R=0.15, G=0.75, B=0.10
  D (Yellow): R=0.50, G=0.45, B=0.05
  E (Orange): R=0.70, G=0.25, B=0.05
  F (Red   ): R=0.80, G=0.15, B=0.05


In [3]:
# Define binding kinetics for stepwise assembly
# k_on matrix: [i,j] = binding rate of i + j (µm²/s)
# k_off matrix: [i,j] = unbinding rate of i-j complex (1/s)

# Initialize matrices (no binding by default)
k_on = np.zeros((n_species, n_species))
k_off = np.zeros((n_species, n_species))

# Binding radius for contact (nm)
# Note: With D_free = 1.5 µm²/s, molecules move ~245 nm per 10 ms frame
# Need binding radius >= 2× RMS displacement to capture encounters
binding_radius = 500  # nm (2× RMS displacement)

# Define stepwise assembly pathway - OPTIMIZED for full complex formation
# Strategy: Fast initial steps → slower final steps for stable endpoint
# Goal: See 1-2 full ABCDEF complexes in 30s

# Step 1: A (0) + B (1) → AB (fast, tight binding)
k_on[0, 1] = k_on[1, 0] = 20.0   # Very fast nucleation (µm²/s)
k_off[0, 1] = k_off[1, 0] = 0.05  # Very stable (τ=20s)

# Step 2: AB + C (2) → ABC (fast, moderately stable)
k_on[1, 2] = k_on[2, 1] = 15.0   # Fast growth
k_off[1, 2] = k_off[2, 1] = 0.08  # Moderately stable (τ=12.5s)

# Step 3: ABC + D (3) → ABCD (moderate)
k_on[2, 3] = k_on[3, 2] = 10.0   # Slower assembly
k_off[2, 3] = k_off[3, 2] = 0.15  # Less stable (τ=6.7s)

# Step 4: ABCD + E (4) → ABCDE (slow, rate-limiting step)
k_on[3, 4] = k_on[4, 3] = 5.0    # Rate-limiting
k_off[3, 4] = k_off[4, 3] = 0.20  # Transient (τ=5s)

# Step 5: ABCDE + F (5) → ABCDEF (slow on, very slow off)
k_on[4, 5] = k_on[5, 4] = 5.0    # Slow final assembly
k_off[4, 5] = k_off[5, 4] = 0.03  # Very stable final product (τ=33s)

print("\nBinding kinetics matrix (k_on in µm²/s):")
print("        ", "  ".join([f"{c:6s}" for c in colors]))
for i, c_i in enumerate(colors):
    print(f"{c_i:6s}: ", "  ".join([f"{k_on[i,j]:6.1f}" for j in range(n_species)]))

print("\nUnbinding kinetics matrix (k_off in 1/s):")
print("        ", "  ".join([f"{c:6s}" for c in colors]))
for i, c_i in enumerate(colors):
    print(f"{c_i:6s}: ", "  ".join([f"{k_off[i,j]:6.2f}" for j in range(n_species)]))

# Calculate bound lifetimes
print(f"\nBound lifetimes (τ = 1/k_off):")
print(f"  A-B:   {1/k_off[0,1]:5.1f} s (very stable)")
print(f"  AB-C:  {1/k_off[1,2]:5.1f} s (stable)")
print(f"  ABC-D: {1/k_off[2,3]:5.1f} s (moderate)")
print(f"  ABCD-E: {1/k_off[3,4]:4.1f} s (transient)")
print(f"  ABCDE-F: {1/k_off[4,5]:4.1f} s (very stable final)")

print(f"\nBinding radius: {binding_radius} nm")
print(f"  (With D = 1.5 µm²/s, RMS displacement per 10ms ≈ 245 nm)")
print(f"  (Binding radius = {binding_radius/245:.1f}× RMS displacement)")

print(f"\n✓ Optimized for observable full complex (ABCDEF) formation in 30s!")


Binding kinetics matrix (k_on in µm²/s):
         Blue    Cyan    Green   Yellow  Orange  Red   
Blue  :     0.0    20.0     0.0     0.0     0.0     0.0
Cyan  :    20.0     0.0    15.0     0.0     0.0     0.0
Green :     0.0    15.0     0.0    10.0     0.0     0.0
Yellow:     0.0     0.0    10.0     0.0     5.0     0.0
Orange:     0.0     0.0     0.0     5.0     0.0     5.0
Red   :     0.0     0.0     0.0     0.0     5.0     0.0

Unbinding kinetics matrix (k_off in 1/s):
         Blue    Cyan    Green   Yellow  Orange  Red   
Blue  :    0.00    0.05    0.00    0.00    0.00    0.00
Cyan  :    0.05    0.00    0.08    0.00    0.00    0.00
Green :    0.00    0.08    0.00    0.15    0.00    0.00
Yellow:    0.00    0.00    0.15    0.00    0.20    0.00
Orange:    0.00    0.00    0.00    0.20    0.00    0.03
Red   :    0.00    0.00    0.00    0.00    0.03    0.00

Bound lifetimes (τ = 1/k_off):
  A-B:    20.0 s (very stable)
  AB-C:   12.5 s (stable)
  ABC-D:   6.7 s (moderate)
  ABCD-E:  5.0

## 2. Initialize Diffusion Simulator with Binding Kinetics

Create a 2D simulation area with realistic parameters:
- 10 µm × 10 µm area
- 10 ms frame time
- Reflective boundaries (molecules stay in field of view)
- Binding kinetics for stepwise assembly

In [4]:
# Import BindingKinetics class
from DiffusionSimulation import BindingKinetics

# Simulation parameters
area = (16000, 9000)  # 15 µm × 15 µm in nm
dt = 20.0              # 10 ms frame time
t_exposure = 20.0      # 10 ms exposure (same as frame time)
sigma0 = 20.0          # 20 nm base localization error
s0 = 150.0             # 150 nm PSF width

# Create binding kinetics object
binding_kinetics = BindingKinetics(
    colors=colors,
    k_on_matrix=k_on,
    k_off_matrix=k_off,
    binding_radius=binding_radius
)

# Initialize simulator with binding kinetics
simulator = DiffusionSimulator2D(
    area=area,
    dt=dt,
    t_exposure=t_exposure,
    sigma0=sigma0,
    s0=s0,
    boundary='reflective',  # Keep molecules in view
    binding_kinetics=binding_kinetics
)

print(f"Simulator initialized:")
print(f"  Area: {area[0]/1000:.1f} × {area[1]/1000:.1f} µm")
print(f"  Frame time: {dt} ms")
print(f"  Localization error: {sigma0} nm")
print(f"  Binding enabled: ✓")

Simulator initialized:
  Area: 16.0 × 9.0 µm
  Frame time: 20.0 ms
  Localization error: 20.0 nm
  Binding enabled: ✓


## 3. Add Molecules to Simulation

Add several copies of each molecule type, distributed throughout the field of view.

**Diffusion coefficients:**
- Free molecules: ~1 µm²/s (typical for small proteins)
- Bound complexes: ~0.2 µm²/s (slower due to larger size)

In [5]:
# Number of molecules per species
n_per_species = 20  # 5 molecules of each type

# Diffusion coefficients (µm²/s) - OPTIMIZED
D_free = 10    # Free diffusion (faster for more encounters)
D_bound = 5   # Bound diffusion (much slower for stability, 15× reduction)

print(f"Diffusion coefficients:")
print(f"  D_free:  {D_free} µm²/s")
print(f"  D_bound: {D_bound} µm²/s")
print(f"  Ratio:   {D_free/D_bound:.1f}× slower when bound")

# RMS displacements per frame
import numpy as np
dt_s = dt / 1000  # Convert ms to s
rms_free = np.sqrt(4 * D_free * 1e6 * dt_s)  # nm
rms_bound = np.sqrt(4 * D_bound * 1e6 * dt_s)  # nm

print(f"\nRMS displacement per frame (dt={dt} ms):")
print(f"  Free:  {rms_free:.0f} nm")
print(f"  Bound: {rms_bound:.0f} nm")
print(f"  → Bound complexes move {D_free/D_bound:.1f}× slower (more stable)")

# Add molecules
molecule_ids = {color: [] for color in colors}

for i, (name, props) in enumerate(molecules.items()):
    color = props['color']
    
    for j in range(n_per_species):
        # Random initial position
        x0 = np.random.uniform(1000, area[0] - 1000)  # nm (avoid edges)
        y0 = np.random.uniform(1000, area[1] - 1000)  # nm
        
        # Add molecule
        mol = simulator.add_molecule(
            color=color,
            position=np.array([x0, y0]),
            D_free=D_free,
            D_bound=D_bound,
            spectral_profile={
                'A_R': props['A_R'],
                'A_G': props['A_G'],
                'A_B': props['A_B']
            }
        )
        
        molecule_ids[color].append(mol.molecule_id)

print(f"\nAdded {len(simulator.molecules)} molecules:")
for color in colors:
    print(f"  {color:6s}: {len(molecule_ids[color])} molecules")

print(f"\n✓ Optimized parameters for observable stepwise assembly!")

Diffusion coefficients:
  D_free:  10 µm²/s
  D_bound: 5 µm²/s
  Ratio:   2.0× slower when bound

RMS displacement per frame (dt=20.0 ms):
  Free:  894 nm
  Bound: 632 nm
  → Bound complexes move 2.0× slower (more stable)

Added 120 molecules:
  Blue  : 20 molecules
  Cyan  : 20 molecules
  Green : 20 molecules
  Yellow: 20 molecules
  Orange: 20 molecules
  Red   : 20 molecules

✓ Optimized parameters for observable stepwise assembly!


## 4. Run Simulation

Simulate for several seconds to observe assembly dynamics.

**Timeline:**
- 0-5s: Initial diffusion and first binding events (A+B)
- 5-10s: Secondary assembly (AB+C, ABC+D)
- 10-20s: Higher-order complexes (ABCD+E, ABCDE+F)
- 20-30s: Steady state with fully assembled complexes

In [6]:
# Simulation duration
duration = 60.0  # seconds
n_frames = int(duration * 1000 / dt)  # Convert to number of frames

print(f"Running simulation...")
print(f"  Duration: {duration} s")
print(f"  Frames: {n_frames} ({dt} ms each)")
print(f"  Total time: {n_frames * dt / 1000:.1f} s")
print()

# Run simulation with binding enabled
simulator.run(
    n_steps=n_frames,
    enable_binding=True
)

# Get binding events
n_bind = len(simulator.binding_kinetics.binding_events)
n_unbind = len(simulator.binding_kinetics.unbinding_events)

print(f"\n✅ Simulation complete!")
print(f"  Trajectories generated for {len(simulator.molecules)} molecules")
print(f"  Binding events: {n_bind}")
print(f"  Unbinding events: {n_unbind}")
print(f"  Total events: {n_bind + n_unbind}")

Running simulation...
  Duration: 60.0 s
  Frames: 3000 (20.0 ms each)
  Total time: 60.0 s


✅ Simulation complete!
  Trajectories generated for 120 molecules
  Binding events: 186
  Unbinding events: 149
  Total events: 335


## 5. Analyze Binding Events

Examine the binding history to see assembly progression.

In [7]:
# Analyze binding events
binding_events = simulator.binding_kinetics.binding_events
unbinding_events = simulator.binding_kinetics.unbinding_events
all_events = binding_events + unbinding_events

if len(all_events) > 0:
    print(f"Binding Events Summary:")
    print(f"  Binding events: {len(binding_events)}")
    print(f"  Unbinding events: {len(unbinding_events)}")
    print(f"  Total events: {len(all_events)}")
    print()
    
    # Show first 10 events
    print(f"First 10 events:")
    
    # Combine and sort all events by time
    combined_events = []
    for event in binding_events:
        combined_events.append({
            'type': 'bind',
            'time': event['time'],
            'mol1_id': event['mol1_id'],
            'mol2_id': event['mol2_id'],
            'mol1_color': event['mol1_color'],
            'mol2_color': event['mol2_color']
        })
    
    for event in unbinding_events:
        combined_events.append({
            'type': 'unbind',
            'time': event['time'],
            'mol1_id': event['mol1_id'],
            'mol2_id': event['mol2_id'],
            'mol1_color': event['mol1_color'],
            'mol2_color': event['mol2_color']
        })
    
    # Sort by time
    combined_events.sort(key=lambda x: x['time'])
    
    for i, event in enumerate(combined_events[:10]):
        time_s = event['time'] / 1000  # Convert ms to s
        
        if event['type'] == 'bind':
            print(f"  [{time_s:6.2f}s] {event['mol1_color']:6s} + {event['mol2_color']:6s} → BOUND")
        else:
            print(f"  [{time_s:6.2f}s] {event['mol1_color']:6s} - {event['mol2_color']:6s} → UNBOUND")
    
    if len(combined_events) > 10:
        print(f"  ... ({len(combined_events) - 10} more events)")
else:
    print("⚠️  No binding events occurred (try longer simulation or higher k_on)")

Binding Events Summary:
  Binding events: 186
  Unbinding events: 149
  Total events: 335

First 10 events:
  [  0.06s] Orange + Red    → BOUND
  [  0.11s] Blue   + Cyan   → BOUND
  [  0.11s] Orange + Red    → BOUND
  [  0.12s] Orange + Red    → BOUND
  [  0.26s] Blue   + Cyan   → BOUND
  [  0.28s] Yellow + Orange → BOUND
  [  0.30s] Blue   + Cyan   → BOUND
  [  0.35s] Orange + Red    → BOUND
  [  0.48s] Blue   + Cyan   → BOUND
  [  0.81s] Cyan   + Green  → BOUND
  ... (325 more events)


## 6. Generate Ground Truth RGB Video

Create a "perfect" visualization showing molecules as colored Gaussians.
When molecules bind, their colors blend together!

In [ ]:
# Initialize camera adapter
adapter = CameraAdapter(simulator)

# Generate RGB video
print("Generating ground truth RGB video...")
print(f"  Rendering {n_frames} frames")
print(f"  Image size: ~{int(area[0]/69)} × {int(area[1]/69)} pixels")
print(f"  Gaussian width: 50 nm (super-resolution appearance)")
print()

# Output path
output_path = 'stepwise_assembly_rgb.tiff'

# Generate video
rgb_video = adapter.generate_ground_truth_rgb_video(
    output_path=output_path,
    frame_indices=np.arange(n_frames),
    pixel_size_nm=69.0/2,           # Camera pixel size
    gaussian_width_nm=50.0,       # Render as ~50 nm Gaussians
    colormap='direct',            # Use spectral profiles as RGB
    save_video=True,
    background_value=2,          # Slight background
    scale_intensity=True,         # Auto-scale per frame
)

print(f"\n✅ RGB video saved to: {output_path}")
print(f"   Shape: {rgb_video.shape} (frames, height, width, RGB)")
print(f"   Dtype: {rgb_video.dtype}")

Generating ground truth RGB video...
  Rendering 3000 frames
  Image size: ~231 × 130 pixels
  Gaussian width: 50 nm (super-resolution appearance)

Generating ground truth RGB video: 3000 frames, 464×261 pixels


## 7. Visualize Sample Frames

Show snapshots at different timepoints to see assembly progression.

In [ ]:
# Plot sample frames
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Timepoints to show (seconds)
timepoints = [0, 5, 10, 15, 20, 25]  # 0s, 5s, 10s, 15s, 20s, 25s
frame_indices = [int(t * 1000 / dt) for t in timepoints]

for idx, (ax, t, f_idx) in enumerate(zip(axes.flat, timepoints, frame_indices)):
    if f_idx < rgb_video.shape[0]:
        ax.imshow(rgb_video[f_idx], origin='lower')
        ax.set_title(f't = {t:.1f} s (frame {f_idx})', fontsize=14, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', 
               transform=ax.transAxes, fontsize=20)
        ax.set_title(f't = {t:.1f} s', fontsize=14)
    
    ax.set_xlabel('X (pixels)', fontsize=12)
    ax.set_ylabel('Y (pixels)', fontsize=12)
    ax.set_aspect('equal')

plt.suptitle('Stepwise Molecular Assembly - Ground Truth RGB Video', 
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/stepwise_assembly_frames.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Sample frames saved to: /tmp/stepwise_assembly_frames.png")

## 8. Analyze Binding State Over Time

Track the number of bound molecules throughout the simulation.

In [ ]:
# Analyze binding state over time
# Count bound molecules at each frame
binding_events = simulator.binding_kinetics.binding_events
unbinding_events = simulator.binding_kinetics.unbinding_events

bound_count = np.zeros(n_frames)

for frame_idx in range(n_frames):
    time_ms = frame_idx * dt
    
    # Track bound pairs as sets to avoid double counting
    bound_pairs = set()
    
    # Process all binding events up to this time
    for bind_event in binding_events:
        if bind_event['time'] <= time_ms:
            mol1_id = bind_event['mol1_id']
            mol2_id = bind_event['mol2_id']
            pair = tuple(sorted([mol1_id, mol2_id]))
            bind_time = bind_event['time']
            
            # Check if this pair has unbound by this time
            is_still_bound = True
            for unbind_event in unbinding_events:
                unbind_mol1 = unbind_event['mol1_id']
                unbind_mol2 = unbind_event['mol2_id']
                unbind_pair = tuple(sorted([unbind_mol1, unbind_mol2]))
                
                if (unbind_pair == pair and 
                    unbind_event['time'] > bind_time and 
                    unbind_event['time'] <= time_ms):
                    # This pair unbound after binding, before current time
                    is_still_bound = False
                    break
            
            if is_still_bound:
                bound_pairs.add(pair)
    
    bound_count[frame_idx] = len(bound_pairs)

# Plot binding progression
fig, ax = plt.subplots(figsize=(12, 6))

time_s = np.arange(n_frames) * dt / 1000
ax.plot(time_s, bound_count, linewidth=2, color='darkblue')
ax.fill_between(time_s, 0, bound_count, alpha=0.3, color='lightblue')

ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel('Number of Bound Pairs', fontsize=14)
ax.set_title('Assembly Progression Over Time', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, time_s[-1])
ax.set_ylim(0, bound_count.max() * 1.1 if bound_count.max() > 0 else 1)

plt.tight_layout()
plt.savefig('/tmp/stepwise_assembly_kinetics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Kinetics plot saved to: /tmp/stepwise_assembly_kinetics.png")
print(f"\nAssembly statistics:")
print(f"  Maximum bound pairs: {int(bound_count.max())}")
print(f"  Average bound pairs: {bound_count.mean():.1f}")
print(f"  Final state: {int(bound_count[-1])} bound pairs")

## 9. Summary and Next Steps

**Outputs Generated:**
1. `/tmp/stepwise_assembly_rgb.tiff` - Ground truth RGB video (3000 frames)
2. `/tmp/stepwise_assembly_frames.png` - Sample frames showing assembly
3. `/tmp/stepwise_assembly_kinetics.png` - Binding progression over time

**What to look for in the RGB video:**
- Individual molecules appear as single colors (Blue, Cyan, Green, Yellow, Orange, Red)
- When molecules bind, their colors blend together
- Bound complexes diffuse more slowly (D_bound < D_free)
- Stepwise assembly creates multi-color complexes

**Next Steps:**
1. **Generate Bayer-filtered TIFF stack** for realistic camera simulation
2. **Extract localizations** from Bayer stack using pyBayerSMLM pipeline
3. **Perform spectral unmixing** to identify molecule types
4. **Track trajectories** to measure binding kinetics
5. **Compare to ground truth** to validate pipeline accuracy

In [ ]:
binding_events = simulator.binding_kinetics.binding_events
unbinding_events = simulator.binding_kinetics.unbinding_events

print("="*80)
print("SIMULATION COMPLETE!")
print("="*80)
print(f"\nGenerated files:")
print(f"  1. {output_path}")
print(f"     - Ground truth RGB video ({n_frames} frames)")
print(f"     - Shows assembly as color mixing")
print(f"\n  2. /tmp/stepwise_assembly_frames.png")
print(f"     - Sample frames at t = 0, 5, 10, 15, 20, 25 s")
print(f"\n  3. /tmp/stepwise_assembly_kinetics.png")
print(f"     - Binding progression over time")
print(f"\nSimulation parameters:")
print(f"  - 6 molecule types (Blue, Cyan, Green, Yellow, Orange, Red)")
print(f"  - {n_per_species} molecules per type ({len(simulator.molecules)} total)")
print(f"  - Stepwise assembly: A+B → AB+C → ABC+D → ABCD+E → ABCDE+F")
print(f"  - Duration: {duration} s ({n_frames} frames)")
print(f"  - Binding events: {len(binding_events)}")
print(f"  - Unbinding events: {len(unbinding_events)}")
print(f"  - Total events: {len(binding_events) + len(unbinding_events)}")
print(f"\nNext: Test with real pyBayerSMLM pipeline!")
print("="*80)